In [1]:
print("Hello Roshan")

Hello Roshan


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
import sys
print(sys.executable)

c:\Users\Asus\AppData\Local\Programs\Python\Python313\python.exe


In [4]:
import pandas as pd
import numpy as np

print(pd.__version__)
print(np.__version__)

3.0.5
2.5.1


In [5]:
import os

print(os.getcwd())

c:\Intership\System_Capacity_Care_Load_Analytics\System_Capacity_Care_Load_Analytics\notebooks


In [6]:
df = pd.read_csv("../data/raw/HHS_Unaccompanied_Alien_Children_Program.csv")

In [7]:
df.head()


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
0,"December 21, 2025",6.0,18.0,11.0,"2,484",14.0
1,"December 18, 2025",11.0,50.0,6.0,"2,472",16.0
2,"December 17, 2025",7.0,31.0,11.0,"2,481",10.0
3,"December 16, 2025",8.0,54.0,15.0,"2,468",9.0
4,"December 15, 2025",11.0,42.0,9.0,"2,470",7.0


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1170 entries, 0 to 1169
Data columns (total 6 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Date                                             720 non-null    str    
 1   Children apprehended and placed in CBP custody*  720 non-null    float64
 2   Children in CBP custody                          720 non-null    float64
 3   Children transferred out of CBP custody          720 non-null    float64
 4   Children in HHS Care                             720 non-null    str    
 5   Children discharged from HHS Care                720 non-null    float64
dtypes: float64(4), str(2)
memory usage: 69.5 KB


In [9]:
df.shape

(1170, 6)

In [10]:
df.describe()

,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children discharged from HHS Care
count,720.000000,720.000000,720.000000,720.000000
mean,93.523611,171.494444,128.668056,173.406944
std,72.646625,126.354965,97.322012,125.702841
min,0.000000,7.000000,0.000000,0.000000
25%,12.000000,36.000000,14.000000,19.750000
50%,99.000000,193.000000,157.000000,181.000000
75%,147.250000,263.250000,199.250000,267.000000
max,333.000000,531.000000,440.000000,505.000000


In [11]:
df.isnull().sum()

Date                                               450
Children apprehended and placed in CBP custody*    450
Children in CBP custody                            450
Children transferred out of CBP custody            450
Children in HHS Care                               450
Children discharged from HHS Care                  450
dtype: int64

In [12]:
df.duplicated().sum()

np.int64(449)

In [13]:
df.duplicated().sum()

np.int64(449)

In [14]:
df["Date"] = pd.to_datetime(df["Date"])

In [15]:
df = df.sort_values("Date").reset_index(drop=True)

In [16]:
print(df.columns.tolist())

['Date', 'Children apprehended and placed in CBP custody*', 'Children in CBP custody', 'Children transferred out of CBP custody', 'Children in HHS Care', 'Children discharged from HHS Care']


In [17]:
invalid_transfer = df[
    df["Children transferred out of CBP custody"] >
    df["Children in CBP custody"]
]

print("Invalid Transfer Records:", len(invalid_transfer))
invalid_transfer

Invalid Transfer Records: 86


,Date,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
3,2023-01-24,47.0,42.0,47.0,"7,433",175.0
4,2023-01-25,20.0,22.0,41.0,"7,538",180.0
9,2023-02-02,15.0,13.0,23.0,"7,879",298.0
22,2023-02-22,107.0,215.0,230.0,"7,978",232.0
23,2023-02-23,101.0,162.0,178.0,"7,914",386.0
...,...,...,...,...,...,...
502,2025-01-30,47.0,42.0,47.0,"3,923",159.0
503,2025-02-02,20.0,22.0,41.0,"3,483",168.0
508,2025-02-09,15.0,13.0,23.0,"2,878",99.0
512,2025-02-13,15.0,10.0,23.0,"2,703",72.0


In [18]:
invalid_discharge = df[
    df["Children discharged from HHS Care*"] >
    df["Children in HHS Care"]
]

print("Invalid Discharge Records:", len(invalid_discharge))
invalid_discharge

KeyError: 'Children discharged from HHS Care*'

In [ ]:
print(df.columns.tolist())

['Date', 'Children apprehended and placed in CBP custody*', 'Children in CBP custody', 'Children transferred out of CBP custody', 'Children in HHS Care', 'Children discharged from HHS Care']


In [ ]:
df.to_csv("../data/processed/cleaned_data.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [19]:
import pandas as pd

# Raw dataset load
df = pd.read_csv(
    "../data/raw/HHS_Unaccompanied_Alien_Children_Program.csv",
    dtype="string"
)

# Clean column names
df.columns = df.columns.str.strip()

# Date
df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

# Remove rows where Date is missing
df = df[df["Date"].notna()].copy()

# Numeric columns
numeric_cols = [
    "Children apprehended and placed in CBP custody*",
    "Children in CBP custody",
    "Children transferred out of CBP custody",
    "Children in HHS Care",
    "Children discharged from HHS Care"
]

# Convert numbers like "2,484" -> 2484
for col in numeric_cols:
    df[col] = (
        df[col]
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# Sort by date
df = df.sort_values("Date").reset_index(drop=True)

# Calculate Total System Load
df["Total System Load"] = (
    df["Children in CBP custody"] +
    df["Children in HHS Care"]
)

# Calculate Net Daily Intake
df["Net Daily Intake"] = (
    df["Children transferred out of CBP custody"] -
    df["Children discharged from HHS Care"]
)

# Care Load Growth Rate
df["Care Load Growth Rate"] = (
    df["Total System Load"].pct_change() * 100
)

# Rolling averages
df["7-Day Rolling Load"] = (
    df["Total System Load"].rolling(7).mean()
)

df["14-Day Rolling Load"] = (
    df["Total System Load"].rolling(14).mean()
)

# Backlog
df["Backlog Indicator"] = (
    df["Net Daily Intake"] > 0
).astype(int)

# Save clean dataset
df.to_csv(
    "../data/processed/analytics_data.csv",
    index=False
)

print("✅ Clean analytics dataset saved!")

✅ Clean analytics dataset saved!


In [20]:
print(
    df[
        [
            "Date",
            "Children in HHS Care",
            "Children discharged from HHS Care",
            "Total System Load",
            "Net Daily Intake"
        ]
    ].head(10).to_string()
)

        Date  Children in HHS Care  Children discharged from HHS Care  Total System Load  Net Daily Intake
0 2023-01-12                  6566                                436               6619              -402
1 2023-01-22                  7122                                227               7171              -188
2 2023-01-23                  7280                                181               7330              -142
3 2023-01-24                  7433                                175               7475              -128
4 2023-01-25                  7538                                180               7560              -139
5 2023-01-29                  7472                                303               7517              -292
6 2023-01-30                  7743                                196               7797              -167
7 2023-01-31                  7803                                158               7839              -122
8 2023-02-01                  7903   

In [1]:
print(df.shape)
print(df.columns.tolist())
print(df.head())

NameError: name 'df' is not defined

In [2]:
import os
import glob

print("Current folder:")
print(os.getcwd())

print("\nProject files:")
for root, dirs, files in os.walk(".."):
    for file in files:
        if file.lower().endswith((".csv", ".xlsx", ".xls")):
            print(os.path.join(root, file))

Current folder:
c:\Intership\System_Capacity_Care_Load_Analytics\System_Capacity_Care_Load_Analytics\notebooks

Project files:
..\data\processed\30_day_forecast.csv
..\data\processed\analytics_data.csv
..\data\processed\cleaned_data.csv
..\data\processed\kpi_summary.csv
..\data\raw\HHS_Unaccompanied_Alien_Children_Program.csv
